# Feature Engineering:

---

In [1]:
import numpy as np

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, OrdinalEncoder

In [2]:
data = pd.read_csv("employee_attrition.csv")

In [3]:
data.drop(columns=['EmployeeID', 'BadgeNumber', 'TrainingHours'], inplace=True)

## Building Pipeline:

In [4]:
X = data.drop(columns=['Attrition'])
y = data['Attrition']

In [5]:
# Encoding Target Column from yes/no to 0/1 ->

encode_target = LabelEncoder()

y = encode_target.fit_transform(y)

In [6]:
workLifeBalance_pipeline = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='most_frequent')),
    ('ordinal', OrdinalEncoder(categories=[['Poor', 'Fair', 'Good', 'Excellent']]))
])

num_impute = Pipeline(steps=[
    ('impute', SimpleImputer(strategy='median'))
])

ordinal_encoder = Pipeline(steps=[
    ('ordinal', OrdinalEncoder(categories=[['Low', 'Average', 'High']]))
])

onehot_encoder = Pipeline(steps=[
    ('onehot', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

In [7]:
preprocessing = ColumnTransformer(transformers=[
    ('transfrom', workLifeBalance_pipeline, ['WorkLifeBalance']),
    ('transform_num', num_impute, ['MonthlyIncome']),
    ('Ordinal_Encoding', ordinal_encoder, ['PerformanceRating']),
    ('OneHot_Encoding', onehot_encoder, ['Department', 'Gender', 'Overtime'])
], remainder='passthrough')

## Extracting Final Data:

In [8]:
X_transformed = preprocessing.fit_transform(X)

feature_names = preprocessing.get_feature_names_out()

X_transformed = pd.DataFrame(X_transformed, columns=feature_names, index=X.index)

y_transformed = pd.DataFrame({'Attrition' : y})

final_data = pd.concat([X_transformed, y_transformed], axis=1)

## Checking Multicollinearity:

In [9]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif = []
for i in range(X_transformed.shape[1]):
    vif.append(variance_inflation_factor(X_transformed, i))

pd.DataFrame({'Features' : X_transformed.columns,
             'VIF' : vif})

,Features,VIF
0,transfrom__WorkLifeBalance,4.721224
1,transform_num__MonthlyIncome,23.983520
2,Ordinal_Encoding__PerformanceRating,4.475054
3,OneHot_Encoding__Department_HR,1.590546
4,OneHot_Encoding__Department_IT,2.655350
5,OneHot_Encoding__Department_Marketing,1.784832
6,OneHot_Encoding__Department_Sales,2.421174
7,OneHot_Encoding__Gender_Male,1.875485
8,OneHot_Encoding__Overtime_Yes,1.541221
9,remainder__Age,9.601338


## Final Dataset:

In [10]:
final_data.to_csv('employee_attrition_preprocessed.csv')